# Training Fuzzy C-Means

Notebook ini menjalankan iterasi FCM sampai konvergen, menyimpan seluruh output training, serta membentuk artifact model untuk tahap testing dan evaluasi.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd()
OUTPUT_DIR = ROOT_DIR / "outputs"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"

FEATURE_COLUMNS = [
    "Usia Ibu",
    "Usia Kehamilan",
    "Gravida",
    "Tekanan Darah",
    "Denyut Jantung Janin",
    "Nilai AFI",
    "Kejadian KPD",
]
CLUSTER_LABELS = ["Mild", "Moderate", "Severe"]
CLUSTER_FILE_LABELS = ["K1 - Mild", "K2 - Moderate", "K3 - Severe"]
PARAMS = {
    "c": 3,
    "m": 2.0,
    "epsilon": 0.0001,
    "max_iter": 100,
    "seed": 42,
}

In [2]:
def load_training_inputs():
    preprocessing = pd.read_csv(OUTPUT_DIR / "output_preprocessing.csv", dtype={"Nomor RM": str})
    detail = pd.read_csv(OUTPUT_DIR / "output_preprocessing_detail.csv", dtype={"Nomor RM": str})
    metadata = json.loads((ARTIFACT_DIR / "preprocessing_metadata.json").read_text(encoding="utf-8"))
    preprocessing["No"] = preprocessing["No"].astype(int)
    detail["No"] = detail["No"].astype(int)
    return preprocessing, detail, metadata


def initialize_membership(n_samples, n_clusters, seed):
    rng = np.random.default_rng(seed)
    membership = rng.random((n_clusters, n_samples))
    membership /= membership.sum(axis=0, keepdims=True)
    return membership


def reorder_clusters(membership, centers):
    order = np.argsort(-centers[:, FEATURE_COLUMNS.index("Nilai AFI")])
    return membership[order], centers[order], order


def compute_centers(data, membership, m):
    weights = membership ** m
    numerator = weights @ data
    denominator = weights.sum(axis=1, keepdims=True)
    return numerator / denominator


def compute_distances(data, centers):
    distances = np.linalg.norm(data[None, :, :] - centers[:, None, :], axis=2)
    return np.maximum(distances, 1e-10)


def update_membership(distances, m):
    exponent = 2.0 / (m - 1.0)
    membership = np.zeros_like(distances)
    for sample_index in range(distances.shape[1]):
        sample_distances = distances[:, sample_index]
        zero_mask = np.isclose(sample_distances, 0.0)
        if zero_mask.any():
            membership[zero_mask, sample_index] = 1.0 / zero_mask.sum()
            continue
        ratios = (sample_distances[:, None] / sample_distances[None, :]) ** exponent
        membership[:, sample_index] = 1.0 / ratios.sum(axis=1)
    return membership


def infer_cluster_labels(membership):
    return [CLUSTER_LABELS[index] for index in np.argmax(membership, axis=0)]


def denormalize(value, min_value, max_value):
    return float((value * (max_value - min_value)) + min_value)


def predict_action(cluster_number, usia_ibu_asli=None, gravida_asli=None):
    if cluster_number == 1:
        return "Konservatif / Induksi / Partus spontan"
    if cluster_number == 2:
        return "NST / Induksi / SC / SCTP"
    return "SC cito / SC"


def interpret_pc(value):
    if value >= 0.80:
        return "Sangat Baik"
    if value >= 0.60:
        return "Baik"
    if value >= 0.40:
        return "Cukup"
    return "Buruk"


def write_iteration_output(base_df, membership, max_delta, iteration, partition_coefficient, objective_function):
    export_df = base_df.copy()
    export_df["Iterasi"] = iteration
    export_df[["K1", "K2", "K3"]] = membership.T
    export_df["Hasil"] = infer_cluster_labels(membership)
    export_df["Fungsi Objektif"] = objective_function
    export_df["PC"] = partition_coefficient
    export_df["Max Selisih"] = max_delta
    export_df["No"] = export_df["No"].astype(int)
    export_df["Nomor RM"] = export_df["Nomor RM"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(6)
    export_df.to_csv(
        OUTPUT_DIR / f"output_iterasi_t{iteration}.csv",
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n",
        float_format="%.6f",
    )


def write_cluster_detail_file(cluster_number, cluster_label, squared_membership, feature_df, output_path):
    lines = [f"Klaster {cluster_number} {cluster_label}", ""]
    denominator_terms = [f"{value:.6f}" for value in squared_membership]
    denominator_value = float(squared_membership.sum())

    for feature_name in FEATURE_COLUMNS:
        feature_values = feature_df[feature_name].to_numpy(dtype=float)
        numerator_terms = [f"({weight:.6f} x {value:.6f})" for weight, value in zip(squared_membership, feature_values)]
        numerator_value = float(np.sum(squared_membership * feature_values))
        center_value = numerator_value / denominator_value if denominator_value else 0.0
        lines.extend(
            [
                f"Fitur: {feature_name}",
                "V_kj = (Σ (μ_ik^2 x x_ij)) / (Σ μ_ik^2)",
                f"Pembilang = {' + '.join(numerator_terms)}",
                f"Pembilang = {numerator_value:.6f}",
                f"Penyebut = {' + '.join(denominator_terms)}",
                f"Penyebut = {denominator_value:.6f}",
                f"Hasil = {center_value:.6f}",
                "",
            ]
        )

    output_path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")

In [3]:
preprocessing_df, detail_df, preprocessing_metadata = load_training_inputs()
base_df = preprocessing_df[["No", "Nomor RM"]].copy()
feature_df = preprocessing_df[FEATURE_COLUMNS].copy()
data = feature_df.to_numpy(dtype=float)

membership = initialize_membership(len(preprocessing_df), PARAMS["c"], PARAMS["seed"])

    # Export Inisialisasi Matrix (Iterasi 0)
    init_df = base_df.copy()
    init_df[["μ1", "μ2", "μ3"]] = membership.T
    init_df["Total"] = init_df[["μ1", "μ2", "μ3"]].sum(axis=1)
    init_df = init_df.round(6)
    init_df.to_csv(OUTPUT_DIR / "output_inisialisasi_matrix.csv", index=False, encoding="utf-8-sig")

    pc_history = []
objective_history = []
delta_history = []
cluster_counts_history = []
centers = None

for iteration in range(1, PARAMS["max_iter"] + 1):
    centers_current = compute_centers(data, membership, PARAMS["m"])
    membership, centers_current, _ = reorder_clusters(membership, centers_current)

    distances = compute_distances(data, centers_current)
    objective_function = float(np.sum((membership ** PARAMS["m"]) * (distances ** 2)))
    updated_membership = update_membership(distances, PARAMS["m"])

    updated_centers = compute_centers(data, updated_membership, PARAMS["m"])
    updated_membership, updated_centers, _ = reorder_clusters(updated_membership, updated_centers)

    max_delta = float(np.max(np.abs(updated_membership - membership)))
    partition_coefficient = float(np.sum(updated_membership ** 2) / len(preprocessing_df))

    write_iteration_output(base_df, updated_membership, max_delta, iteration, partition_coefficient, objective_function)
    iteration_labels = infer_cluster_labels(updated_membership)
    iteration_counts = pd.Series(iteration_labels).value_counts().reindex(CLUSTER_LABELS, fill_value=0).to_dict()
    pc_history.append(partition_coefficient)
    objective_history.append(objective_function)
    delta_history.append(max_delta)
    cluster_counts_history.append(iteration_counts)

    membership = updated_membership
    centers = updated_centers

    if max_delta < PARAMS["epsilon"]:
        break

final_distances = compute_distances(data, centers)
final_squared = membership ** 2
final_labels = infer_cluster_labels(membership)

usia_min = preprocessing_metadata["min_values"]["Usia Ibu"]
usia_max = preprocessing_metadata["max_values"]["Usia Ibu"]
usia_asli = preprocessing_df["Usia Ibu"].apply(lambda value: denormalize(value, usia_min, usia_max))
gravidas_asli = detail_df["Gravida Asli"].fillna(0)

prediksi_tindakan = [
    predict_action(index + 1, usia_asli.iloc[row_index], gravidas_asli.iloc[row_index])
    for row_index, index in enumerate(np.argmax(membership, axis=0))
]

output_keanggotaan = base_df.copy()
output_keanggotaan[["μ1", "μ2", "μ3"]] = membership.T
output_keanggotaan = output_keanggotaan.round(6)
output_keanggotaan.to_csv(OUTPUT_DIR / "output_keanggotaan.csv", index=False, encoding="utf-8-sig")

output_euclidean = base_df.copy()
output_euclidean[["d1", "d2", "d3"]] = final_distances.T
output_euclidean = output_euclidean.round(6)
output_euclidean.to_csv(OUTPUT_DIR / "output_euclidean.csv", index=False, encoding="utf-8-sig")

output_partition = base_df.copy()
output_partition[["K1", "K2", "K3"]] = final_squared.T
output_partition = output_partition.round(6)
output_partition.to_csv(OUTPUT_DIR / "output_partition_coefficient.csv", index=False, encoding="utf-8-sig")

output_centers = pd.DataFrame(centers, columns=FEATURE_COLUMNS)
output_centers.insert(0, "Klaster", CLUSTER_FILE_LABELS)
output_centers = output_centers.round(6)
output_centers.to_csv(OUTPUT_DIR / "output_pusat_klaster.csv", index=False, encoding="utf-8-sig")

for cluster_index, cluster_label in enumerate(CLUSTER_LABELS, start=1):
    write_cluster_detail_file(
        cluster_number=cluster_index,
        cluster_label=cluster_label,
        squared_membership=final_squared[cluster_index - 1],
        feature_df=feature_df,
        output_path=OUTPUT_DIR / f"output_pusat_klaster_K{cluster_index}.txt",
    )

output_final = base_df.copy()
output_final[["K1", "K2", "K3"]] = membership.T
output_final[["μ1²", "μ2²", "μ3²"]] = final_squared.T
output_final["Hasil"] = final_labels
output_final["Prediksi Tindakan"] = prediksi_tindakan
output_final["Tindakan Aktual"] = detail_df["Tindakan"].fillna("")
for column in ["K1", "K2", "K3", "μ1²", "μ2²", "μ3²"]:
    output_final[column] = pd.to_numeric(output_final[column], errors="coerce")
output_final["No"] = output_final["No"].astype(int)
output_final["Nomor RM"] = output_final["Nomor RM"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(6)
output_final.to_csv(
    OUTPUT_DIR / "output_fcm_final.csv",
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
    float_format="%.6f",
)

fcm_model = {
    "feature_columns": FEATURE_COLUMNS,
    "cluster_labels": {str(index + 1): label for index, label in enumerate(CLUSTER_LABELS)},
    "min_values": preprocessing_metadata["min_values"],
    "max_values": preprocessing_metadata["max_values"],
    "centers": centers.tolist(),
    "params": PARAMS,
    "pc_final": pc_history[-1],
    "objective_final": objective_history[-1],
    "iterations_run": len(pc_history),
}
(ARTIFACT_DIR / "fcm_model.json").write_text(json.dumps(fcm_model, indent=2), encoding="utf-8")

training_summary = {
    "valid_rows": int(len(preprocessing_df)),
    "iterations_run": len(pc_history),
    "pc_final": pc_history[-1],
    "objective_final": objective_history[-1],
    "cluster_counts": pd.Series(final_labels).value_counts().reindex(CLUSTER_LABELS, fill_value=0).to_dict(),
}
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(training_summary, indent=2), encoding="utf-8")

iteration_summary = pd.DataFrame(
    {
        "Iterasi": list(range(1, len(pc_history) + 1)),
        "PC": pc_history,
        "Fungsi Objektif": objective_history,
        "Interpretasi": [interpret_pc(value) for value in pc_history],
        "Max Selisih": delta_history,
    }
)
for cluster_label in CLUSTER_LABELS:
    iteration_summary[cluster_label] = [int(counts[cluster_label]) for counts in cluster_counts_history]
iteration_summary.to_csv(
    OUTPUT_DIR / "output_ringkasan_iterasi.csv",
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
    float_format="%.6f",
)

objective_summary = pd.DataFrame({
    "Iterasi": list(range(1, len(objective_history) + 1)),
    "Fungsi Objektif": objective_history
})
objective_summary.to_csv(
    OUTPUT_DIR / "output_fungsi_objektif.csv",
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
    float_format="%.6f",
)

pc_artifact = {
    "pc_history": pc_history,
    "objective_history": objective_history,
    "delta_history": delta_history,
    "iterations_run": len(pc_history),
    "pc_final": pc_history[-1],
    "objective_final": objective_history[-1],
}
(ARTIFACT_DIR / "pc_history.json").write_text(json.dumps(pc_artifact, indent=2), encoding="utf-8")

print(f"Iterasi selesai : {len(pc_history)}")
print(f"PC final        : {pc_history[-1]:.6f}")
print(f"J final         : {objective_history[-1]:.6f}")
print("Semua output training berhasil diperbarui.")
output_final.head()

Iterasi selesai : 26
PC final        : 0.643495
Semua output training berhasil diperbarui.


,No,Nomor RM,K1,K2,K3,μ1²,μ2²,μ3²,Hasil,Prediksi Tindakan,Tindakan Aktual
0,1,054121,0.066906,0.891390,0.041704,0.004476,0.794576,0.001739,Moderate,NST / Induksi / SC / SCTP,sc
1,2,065046,0.175207,0.707306,0.117486,0.030698,0.500282,0.013803,Moderate,NST / Induksi / SC / SCTP,sc
2,3,078533,0.067389,0.889176,0.043435,0.004541,0.790633,0.001887,Moderate,NST / Induksi / SC / SCTP,sc
3,4,083531,0.889121,0.052981,0.057899,0.790536,0.002807,0.003352,Mild,Konservatif / Induksi / Partus spontan,sc
4,5,079120,0.183401,0.408590,0.408009,0.033636,0.166946,0.166471,Moderate,NST / Induksi / SC / SCTP,sc
